# BERT Demo

## problem statement
+ predict sentiment of given statement
+ sentiment +ve or negative

### install required pkgs

In [1]:
# used to download model from hugginface
# pretrained model trained on genric data so, we finetune on our data
!pip install transformers


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### import required pkgs

In [8]:
import numpy as np
import torch
from transformers import BertTokenizer,BertForSequenceClassification

### select device


In [3]:
device = ""
if torch.cuda.is_available():
    device = "cuda"
elif torch.mps.is_available():
    device = 'mps'
else:
    device='cpu'

print(device)

cuda


### get data

In [4]:
texts = [
    "A masterpiece from start to finish! Beautifully crafted and emotionally resonant.",
    "Laughed, cried, cheered them on. A beautifully crafted film that will leave you feeling uplifted.",
    "Action-packed with heart! 10/10 would watch again!",
    "Original and deep. Stunning visuals and themes that stay with you.",
    "One of the best movies I've seen in a long time! Outstanding performances and perfect emotional resonance.",
    "Humor, heart, excitement. Strong performances make it easy to invest.",
    "Brilliant direction, clever script, and an ensemble cast that shines. A masterpiece!",
    "A game-changer. Thought-provoking, visually stunning, and memorable performances.",
    "Pacing is spot-on, dialogue is witty, and themes are timely and important.",
    "Walked in skeptical, left feeling inspired! A must-watch with a clear message and thrilling action scenes.",
    "Clichéd plot and unoriginal ideas. Boring and overhyped.",
    "Underwhelming experience. Weak characters and poor pacing.",
    "Waste of time. Predictable and lacking in depth. Disappointing!",
    "Expected more from the director, but this was a letdown. Nothing special here.",
    "Confusing and disjointed. Couldn't care less about the characters or story.",
    "Lacked tension and suspense. Unengaging and forgettable.",
    "Not as advertised. Promised thrills, delivered disappointment. A misfire!",
    "Awful dialogue and cringeworthy acting. Avoid this one at all costs!",
    "Underdeveloped characters and a convoluted plot. Not worth the time.",
    "Felt like I was watching paint dry. Boring, unoriginal, and poorly made."
]

In [7]:
# decides labels
# 1 : positive
# 0 : Negative
labels = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

### use tokanizer

In [49]:
# get tokenizer for bert-uncased model
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

### load the pre-trained model

In [50]:
# load bert-base-uncased model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased')

# move the model to the selected device
model = model.to(device) 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### train the model on our data

In [51]:
# define the hyper parameter
epochs = 10
learning_rate = 0.0001
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

In [52]:
# enable the training 
model.train()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [53]:
# # start the training loops
# for epoch in range(epochs):

#     # collect total loss
#     total_loss = 0

#     # get the review(text) along with the label and pass it down to the model
#     for text, label in zip(texts,labels):

#         # convert the text into tokens using tokenizer
#         intput_tokens = tokenizer(
#             text,
#             return_tensors = "pt",
#             truncation = True,
#             padding=True,
#             max_length = 32
#         )

#         # move all tokens to device
#         # this dictonary have form : <index>: <token moved to device>
#         intput_tokens = {index: token.to(device) for index, token in intput_tokens.items()}

#         # convert labels to tensor and move them to seleted device 
#         labels_tensor = torch.tensor([label]).to(device)

#         #clear gradient
#         optimzer.zero_grad()

#         # predict output
#         # splat operator : unpacking operator
#         output = model(**intput_tokens, labels=labels_tensor)

#         # get loss
#         loss = output.loss

#         # backpropogation
#         loss.backward()

#         # update parameters
#         optimzer.step()

#         # collect loss
#         total_loss += loss.item()

#     print(f"epcoh = {epoch}, loss = {total_loss}")


# start the training loop
for epoch in range(epochs):

    # collect total loss
    total_loss = 0

    # get the review along with the label and pass it down to the model
    for text, label in zip(texts, labels):

        # convert the text into tokens using tokenizer
        input_tokens = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=32
        )

        # move all the tokens to selected device
        # this dictionary will have form: <index>: <token moved to device>
        input_tokens = {index: token.to(device) for index, token in input_tokens.items()}

        # convert the labels to tensors and move them to selected device
        labels_tensor = torch.tensor([label]).to(device)

        # clear the gradients
        optimizer.zero_grad()

        # predict the output
        # splat operator
        outputs = model(**input_tokens, labels=labels_tensor)

        # get the loss
        loss = outputs.loss

        # backpropagation
        loss.backward()

        # update parameters
        optimizer.step()

        # collect the loss
        total_loss += loss.item()

    print(f"epoch = {epoch}, loss = {total_loss}")



epoch = 0, loss = 18.658784586936235
epoch = 1, loss = 23.05332186818123
epoch = 2, loss = 16.776982352137566
epoch = 3, loss = 18.211601108312607
epoch = 4, loss = 17.079593002796173
epoch = 5, loss = 20.29327154159546
epoch = 6, loss = 16.645824372768402
epoch = 7, loss = 14.670357674360275
epoch = 8, loss = 15.921789228916168
epoch = 9, loss = 14.268445283174515


In [113]:
# predict sentiment of new review
review = "total waste of money"

In [114]:
# convert text into tokens using tokenizer
input_tokens = tokenizer(
            review,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=32
        )

In [115]:
# move all tokens to device
in_tokens = {index: token.to(device) for index, token in input_tokens.items()}

In [116]:
# predict sentiment
with torch.no_grad():
    outputs = model(**in_tokens)

In [117]:
# using sigmoid we get 1 or value
# but this is not sigmoid its Linear we get 2 value

print(outputs.logits)

tensor([[0.1955, 0.0674]], device='cuda:0')


In [118]:
# to find index of final value
prediction = torch.argmax(outputs.logits,dim=1).item()
prediction

0

In [119]:
if prediction == 1:
    print("review is positive ")
else:
    print("review is negative")

review is negative
